# Cancer Image Detection With PyTorch

## Problem Statement
Millions in developing countries like Bangladesh
lack early cancer diagnosis. This project uses
CNN to detect cancerous patterns in medical images.

## Tech Stack
- PyTorch
- CNN Architecture  
- Python
- Google Colab

## Certificate
IBM SkillsNetwork — May 15, 2026

In [ ]:
# Step 1: Install required libraries
!pip install torch torchvision matplotlib numpy

# Import libraries
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Dataset
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import os

print("✅ All libraries imported successfully!")
print(f"PyTorch version: {torch.__version__}")

✅ All libraries imported successfully!
PyTorch version: 2.11.0+cu128


In [ ]:
# Step 2: Download Cancer Image Dataset from Kaggle
# We use BreakHis dataset (breast cancer histology images)
# Or use this simple approach with torchvision

# For demo — using simulated medical image data
# In real project, replace with actual cancer dataset

import torchvision.datasets as datasets

# Define transforms
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

print("✅ Data transforms defined!")
print("Transform pipeline:")
print(transform)

✅ Data transforms defined!
Transform pipeline:
Compose(
    Resize(size=(224, 224), interpolation=bilinear, max_size=None, antialias=True)
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
)


In [ ]:
# Step 3: Create Custom Dataset Class

class CancerImageDataset(Dataset):
    def __init__(self, data, labels, transform=None):
        self.data = data
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        image = self.data[idx]
        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)

        return image, label

# Create simulated dataset for demonstration
# Replace with real cancer image dataset

import torch
from torchvision import datasets, transforms
from torch.utils.data import random_split

# Download sample dataset (CIFAR-10 as placeholder)
full_dataset = datasets.CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform=transform
)

# Split dataset
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(
    full_dataset, [train_size, val_size]
)

# Create DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False
)

print(f"✅ Dataset loaded!")
print(f"Training samples: {train_size}")
print(f"Validation samples: {val_size}")

✅ Dataset loaded!
Training samples: 40000
Validation samples: 10000


In [ ]:
# Step 4: Build CNN Model for Cancer Detection

class CancerDetectionCNN(nn.Module):
    def __init__(self, num_classes=2):
        super(CancerDetectionCNN, self).__init__()

        # Convolutional layers
        self.conv_layers = nn.Sequential(
            # Conv Block 1
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            # Conv Block 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            # Conv Block 3
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
        )

        # Fully connected layers
        self.fc_layers = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 28 * 28, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = self.fc_layers(x)
        return x

# Initialize model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = CancerDetectionCNN(num_classes=2).to(device)

print("✅ CNN Model created!")
print(f"Device: {device}")
print(f"\nModel Architecture:")
print(model)

AcceleratorError: CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
# Step 5: Train the Model

# Loss function & optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.StepLR(
    optimizer, step_size=5, gamma=0.1
)

# Training function
def train_model(model, train_loader, val_loader, epochs=10):
    train_losses = []
    val_losses = []
    train_accuracies = []
    val_accuracies = []

    for epoch in range(epochs):
        # Training phase
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        train_loss = running_loss / len(train_loader)
        train_acc = 100 * correct / total
        train_losses.append(train_loss)
        train_accuracies.append(train_acc)

        # Validation phase
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(device)
                labels = labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                _, predicted = torch.max(outputs, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()

        val_loss = val_loss / len(val_loader)
        val_acc = 100 * val_correct / val_total
        val_losses.append(val_loss)
        val_accuracies.append(val_acc)

        print(f"Epoch [{epoch+1}/{epochs}]")
        print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
        print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%")
        print("-" * 50)

        scheduler.step()

    return train_losses, val_losses, train_accuracies, val_accuracies

# Run training
print("🚀 Starting Training...")
train_losses, val_losses, train_accs, val_accs = train_model(
    model, train_loader, val_loader, epochs=10
)
print("✅ Training Complete!")



In [ ]:
# Step 6: Visualize Results

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss plot
axes[0].plot(train_losses, label='Train Loss', color='blue')
axes[0].plot(val_losses, label='Val Loss', color='red')
axes[0].set_title('Training vs Validation Loss')
axes[0].set_xlabel('Epochs')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True)

# Accuracy plot
axes[1].plot(train_accs, label='Train Accuracy', color='blue')
axes[1].plot(val_accs, label='Val Accuracy', color='red')
axes[1].set_title('Training vs Validation Accuracy')
axes[1].set_xlabel('Epochs')
axes[1].set_ylabel('Accuracy (%)')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig('training_results.png')
plt.show()
print("✅ Results visualized!")

In [ ]:
# Step 7: Save Model

torch.save(model.state_dict(), 'cancer_detection_model.pth')
print("✅ Model saved as 'cancer_detection_model.pth'")

# Final Summary
print("\n" + "="*50)
print("🎯 CANCER IMAGE DETECTION MODEL — SUMMARY")
print("="*50)
print(f"Final Train Accuracy: {train_accs[-1]:.2f}%")
print(f"Final Val Accuracy: {val_accs[-1]:.2f}%")
print(f"Model: CNN with 3 Conv Blocks")
print(f"Framework: PyTorch")
print(f"Device: {device}")
print("="*50)
print("Built by: Rayhan Uddin")
print("Purpose: Cancer Detection using ML")
print("="*50)